# Wildfire Ignition Risk Model — 6-Year Rolling Water Year (Monthly)

**Purpose:** Train a rolling XGBoost classifier to predict monthly wildfire
ignition risk. For each prediction year, the model is trained on the
6 preceding water years, then evaluated on the target year.

**Training scheme:**
- Predict water year `Y` using training data from water years `Y-7` to `Y-1`
- Probability calibration via `CalibratedClassifierCV` on the held-out 10% calibration set

**Features used:**

| Group | Features |
|-------|----------|
| Weather | `dead_fuel_moisture_1000hr/100hr`, `max/min_air_temperature`, `max/min_relative_humidity`, `precipitation_amount`, `specific_humidity`, `surface_downwelling_shortwave_flux_in_air`, `wind_speed`, `SWE`, `LAI` |
| Wind direction | `wind_direction_category_N/NE/E/SE/S/SW/W/NW` (mode → one-hot) |
| Vegetation | `veg_group_Chaparral/Conifer Alpine/Conifer Forest/Grassland/Oak Woodland/Shrub` |
| Terrain | `slope_max`, `slope_avg` |
| Infrastructure | `road_density_km_km2`, `minor_line_density`, `trans_line_density` |
| Label / lag | `IS_FIRE`, `prev_fire` |

> **Excluded:** `pole_density`, `tower_density` (removed), `pdsi` (commented out),
> `population_density`, `transformer_density`

## 0. Configuration

Edit paths and experiment settings here only.

In [80]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input data
DATA_VERSION  = 'Monthly_03152026_calibration'
DATA_DIR      = os.path.join(PROJECT_ROOT, 'Clean_Data', 'Model_Data',
                             'Evaluation', 'Features_w_Label', DATA_VERSION)

# Model + prediction output
LEN_TRAIN       = 6               # number of training water years
EXPERIMENT_NAME = 'all_power' # tag for this run
PREDICT_YEARS   = range(2001, 2021)
RANDOM_STATE    = 42

MODEL_DIR = os.path.join(PROJECT_ROOT, 'Model',
                         f'Monthly_{LEN_TRAIN}_year_{EXPERIMENT_NAME}')
PRED_DIR  = os.path.join(PROJECT_ROOT, 'Clean_Data', 'Model_Data',
                         'Evaluation', 'Features_w_Label_w_pred',
                         f'Monthly_{LEN_TRAIN}_year_{EXPERIMENT_NAME}',
                         'Calibration_Fixed_2001_2020')
LOG_DIR   = os.path.join(PROJECT_ROOT, 'Logs', 'Model')

for d in [MODEL_DIR, PRED_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Data dir     : {DATA_DIR}")
print(f"Model dir    : {MODEL_DIR}")
print(f"Pred dir     : {PRED_DIR}")
for label, path in [('DATA_DIR', DATA_DIR)]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

Data dir     : E:\zcao\CA_Wildfire\Clean_Data\Model_Data\Evaluation\Features_w_Label\Monthly_03152026_calibration
Model dir    : E:\zcao\CA_Wildfire\Model\Monthly_6_year_all_power
Pred dir     : E:\zcao\CA_Wildfire\Clean_Data\Model_Data\Evaluation\Features_w_Label_w_pred\Monthly_6_year_all_power\Calibration_Fixed_2001_2020
  [OK] DATA_DIR


## 1. Environment Setup

In [81]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                             precision_recall_curve, auc)

gc.collect()
print(f"Python  : {sys.version.split('|')[0].strip()}")
print(f"pandas  : {pd.__version__}")
print(f"xgboost : {xgb.__version__}")

Python  : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas  : 2.2.2
xgboost : 2.1.4


## 2. Helper Functions

- `train_model` — fit XGBoost on training data
- `calculate_precision_recall` — compute P/R/F1 at a given threshold
- `get_water_year_range` — derive training window for a target year

In [82]:
def train_model(train_data, features, label_col):
    X_train = train_data[features]
    y_train = train_data[label_col]
    # train the model
    model = xgb.XGBClassifier(eval_metric='logloss', tree_method='hist')
    model.fit(X_train, y_train)
    return model

# define function to calculate precision and recall based on a threshold
def calculate_precision_recall(y_true, y_pred_proba, threshold, print_output=False):
    y_pred = (y_pred_proba > threshold).astype(int)
    confusion = confusion_matrix(y_true, y_pred)
    precision = confusion[1, 1] / (confusion[1, 1] + confusion[0, 1])
    recall = confusion[1, 1] / (confusion[1, 1] + confusion[1, 0])
    # F1 score
    f1 = 2 * (precision * recall) / (precision + recall)
    if print_output:
        print(f'Threshold: {threshold:.2f}')
        print(f'Precision: {precision * 100:.2f}%')
        print(f'Recall: {recall * 100:.2f}%')
        print("Confusion Matrix")
        print(pd.DataFrame(confusion, index=['True Neg', 'True Pos'], columns=['Pred Neg', 'Pred Pos']))
    # get TP, TN, FP, FN
    TP = confusion[1, 1]
    TN = confusion[0, 0]
    FP = confusion[0, 1]
    FN = confusion[1, 0]
    return TP, TN, FP, FN, precision, recall, f1

def evaluate_model(model, test_data, features, label_col):
    X_test = test_data[features]
    y_test = test_data[label_col]
    # predict the probability of fire
    y_pred = model.predict_proba(X_test)[:, 1]
    # add y_pred to test_data
    test_data['predictions'] = y_pred
    # calculate the roc_auc_score
    roc_auc = roc_auc_score(y_test, y_pred)
    # print roc_auc in a sentence
    # print(f"ROC AUC: {roc_auc:.2f}")
    # Calculate precision and recall values
    precision, recall, _ = precision_recall_curve(y_test, y_pred)
    # Calculate the area under the precision-recall curve
    auc_pr = auc(recall, precision)
    # print(f"Area Under Precision-Recall Curve (AUC-PR): {auc_pr:.2f}")
    # calculate precision and recall at thresholds 0.5
    TP, TN, FP, FN, precision5, recall5, f15 = calculate_precision_recall(y_test, y_pred, 0.5)
    return test_data, roc_auc, auc_pr, TP, TN, FP, FN, precision5, recall5, f15

In [83]:
def get_water_year_range(target_year, num_years=6):
    min_year = target_year - num_years - 1
    min_day = f"{min_year}-10"
    max_day = f"{target_year-1}-09"
    return min_day, max_day

# Example: Get range for Water Year 2007
target_year = 2001
min_day, max_day = get_water_year_range(target_year, num_years=6)

print(f"Predict Water Years {target_year} using training data: {min_day} ~ {max_day}")

Predict Water Years 2001 using training data: 1994-10 ~ 2000-09


## 3. Load Data

Load the calibration (10%) and non-calibration (90%) splits.
Rename octant count columns to `*_freq` to distinguish them from the
one-hot mode columns produced by `pd.get_dummies`.

In [84]:
cali_data = pd.read_parquet(os.path.join(DATA_DIR,
                            'monthly_model_data_by_year_calibration.parquet'))
mod_Human = pd.read_parquet(os.path.join(DATA_DIR,
                            'monthly_model_data_by_year_non_calibration.parquet'))

# Rename octant day-count columns to avoid clash with one-hot dummies
col_to_rename = [f'wind_direction_category_{d}'
                 for d in ['N','NE','E','SE','S','SW','W','NW']]
rename_map = {c: f'{c}_freq' for c in col_to_rename}
cali_data = cali_data.rename(columns=rename_map)
mod_Human = mod_Human.rename(columns=rename_map)

print(f"Calibration     : {cali_data.shape}  fire rate: {cali_data['IS_FIRE'].mean():.4f}")
print(f"Non-calibration : {mod_Human.shape}  fire rate: {mod_Human['IS_FIRE'].mean():.4f}")

Calibration     : (377356, 68)  fire rate: 0.0194
Non-calibration : (3396205, 68)  fire rate: 0.0194


## 4. One-Hot Encode Categorical Columns

Encode `veg_group` and `wind_direction_category` (mode column) into dummy variables.

In [85]:
cat_columns = ['veg_group', 'wind_direction_category']
cali_data = pd.get_dummies(cali_data, columns=cat_columns)
mod_Human = pd.get_dummies(mod_Human, columns=cat_columns)

print(f"Columns after encoding: {mod_Human.shape[1]}")

Columns after encoding: 80


## 5. Feature List

`pole_density` and `tower_density` removed from this run.
All other infrastructure features retained.

In [93]:
FEATURES = [
    # Weather
    'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_100hr',
    'max_air_temperature', 'min_air_temperature',
    'max_relative_humidity', 'min_relative_humidity',
    'precipitation_amount', 'specific_humidity',
    'surface_downwelling_shortwave_flux_in_air',
    'wind_speed', 'SWE',
    # Wind direction (mode → one-hot)
    'wind_direction_category_N',  'wind_direction_category_NE',
    'wind_direction_category_E',  'wind_direction_category_SE',
    'wind_direction_category_S',  'wind_direction_category_SW',
    'wind_direction_category_W',  'wind_direction_category_NW',
    # Vegetation (one-hot)
    'veg_group_Chaparral', 'veg_group_Conifer Alpine',
    'veg_group_Conifer Forest', 'veg_group_Grassland',
    'veg_group_Oak Woodland', 'veg_group_Shrub',
    # Other
    'LAI', 'slope_max', 'slope_avg',
    #'pdsi',  # excluded
    'road_density_km_km2',
    #'line_density_km_per_cell', # Transmission line data from CEC
    'prev_fire',
    'minor_line_density',
    'pole_density',    
    'tower_density',  
    'transformer_density',
    'trans_line_density'
]

print(f"Total features: {len(FEATURES)}")
# Verify all features exist in the data
missing_feats = [f for f in FEATURES if f not in mod_Human.columns]
if missing_feats:
    print(f"WARNING — missing from data: {missing_feats}")
else:
    print("All features present in data.")

Total features: 35
All features present in data.


## 6. Rolling Water-Year Training Loop

For each prediction year in `PREDICT_YEARS`:
1. Define training window: 6 water years ending at `Y-1`
2. Train XGBoost on the full training window (no downsampling — all rows used)
3. Calibrate probabilities via `CalibratedClassifierCV(method='sigmoid')` on the held-out 10% calibration set
4. Predict on target year; compute ROC-AUC, AUC-PR, confusion matrix
5. Save model and predictions

In [94]:
# Earliest available year_month in training data
# Used to skip years that lack sufficient training history
min_year_month = mod_Human['year_month'].min().strftime('%Y-%m')
print(f"Earliest year_month in non-calibration data: {min_year_month}")

Earliest year_month in non-calibration data: 1994-01


In [95]:
len_train = LEN_TRAIN
experiment_name = EXPERIMENT_NAME
results = []
log_messages = []
log_messages.append(f"Num of Year Experiment: {len_train} years")
log_messages.append(f"Experiment Name: {experiment_name}")
# add log to record the current time
log_messages.append(f"Start time: {pd.Timestamp.now()}")
# log the features used in the model
log_messages.append(f"Features used: {FEATURES}")
# Define the range of years to predict
years = PREDICT_YEARS


model_version = f'Monthly_{len_train}_year_{experiment_name}'
model_path = MODEL_DIR
save_predictions_path = PRED_DIR  

# surpass the warning
warnings.filterwarnings("ignore")
cali_data_fixed = cali_data[cali_data['water_year'] >= 2001]
log_messages.append(f"Calibration Data Day min: {cali_data_fixed['year_month'].min()}, max: {cali_data_fixed['year_month'].max()}")
# Iterate over the years with a progress bar
for year in tqdm(years, desc="Processing years"):
    log_messages.append("-" * 50)
    # log current water year
    log_messages.append(f"Processing Water Year: {year}")
    # read eval data, prev-year Oct - current year Sep
    Eval_Human = mod_Human[mod_Human['water_year'] == year]

    # get water year range
    min_day, max_day = get_water_year_range(year, num_years=len_train)
    # if min_day < min_year_month, skip this year
    if min_day < min_year_month:
        log_messages.append(f"Skipping Water Year: {year} due to insufficient training data")
        continue
    # filter the training data
    train_data = mod_Human[(mod_Human['year_month'] >= min_day) & (mod_Human['year_month'] <= max_day)]

    log_messages.append(f"Training Data Day min: {train_data['year_month'].min()}, max: {train_data['year_month'].max()}")
    log_messages.append(f"Eval Data Day min: {Eval_Human['year_month'].min()}, max: {Eval_Human['year_month'].max()}")

    label_col = 'IS_FIRE'
    base_model = train_model(train_data, FEATURES, label_col)

    # Calibrate using validation set
    calibrated_model = CalibratedClassifierCV(base_model, method='sigmoid', cv='prefit')
    calibrated_model.fit(cali_data_fixed[FEATURES], cali_data_fixed[label_col])
    # save model to a pickle file
    with open(f'{model_path}/predict_{year}_model.pkl', 'wb') as f:
         pickle.dump(calibrated_model, f)

    # evaluate the model
    Eval_Human_w_pred, roc_auc, auc_pr, TP, TN, FP, FN, precision5, recall5, f15 = evaluate_model(calibrated_model, Eval_Human, FEATURES, label_col)
    # append the results to the list
    results.append([year, roc_auc, auc_pr, TP, TN, FP, FN, precision5, recall5, f15])

    # add predictions to Eval_Human
    # Eval_Human['predictions'] = calibrated_model.predict_proba(Eval_Human[features])[:, 1]
    # save the predictions to a parquet file
    Eval_Human_w_pred.to_parquet(f'{save_predictions_path}/{year}_predictions.parquet', index=False)

    # clean up the dataframes
    del train_data
    del Eval_Human

    # clean the cache
    gc.collect()

Processing years: 100%|██████████| 20/20 [01:33<00:00,  4.67s/it]


## 7. Save Log

In [96]:
log_path = os.path.join(LOG_DIR, f'{model_version}_log.txt')
with open(log_path, 'w') as f:
    for msg in log_messages:
        f.write(msg + '\n')
print(f"Log saved -> {log_path}")

Log saved -> E:\zcao\CA_Wildfire\Logs\Model\Monthly_6_year_all_power_log.txt


## 8. Results

In [97]:
results_pd = pd.DataFrame(
    results,
    columns=['Year','ROC_AUC','AUC_PR','TP','TN','FP','FN',
             'Precision_0.5','Recall_0.5','F1_0.5']
)
results_pd

,Year,ROC_AUC,AUC_PR,TP,TN,FP,FN,Precision_0.5,Recall_0.5,F1_0.5
0,2001,0.846662,0.151612,126,124202,214,2608,0.370588,0.046086,0.081978
1,2002,0.845988,0.144314,110,124599,194,2647,0.361842,0.039898,0.071872
2,2003,0.846551,0.150683,127,124650,165,2328,0.434932,0.051731,0.092465
3,2004,0.846783,0.154660,131,124458,189,2700,0.409375,0.046273,0.083148
4,2005,0.868207,0.169526,118,124891,126,2225,0.483607,0.050363,0.091225
5,2006,0.864348,0.184637,142,124216,199,2739,0.416422,0.049288,0.088144
6,2007,0.846239,0.207816,161,122853,164,3377,0.495385,0.045506,0.083355
7,2008,0.839786,0.194246,225,123309,182,2916,0.552826,0.071633,0.126832
8,2009,0.854142,0.166847,146,124422,180,2234,0.447853,0.061345,0.107908
9,2010,0.867668,0.161262,123,124872,140,1904,0.467681,0.060681,0.107424


In [98]:
avg_roc_auc = results_pd['ROC_AUC'].mean()
avg_auc_pr  = results_pd['AUC_PR'].mean()
print(f"Average ROC-AUC : {avg_roc_auc:.4f}")
print(f"Average AUC-PR  : {avg_auc_pr:.4f}")

Average ROC-AUC : 0.8461
Average AUC-PR  : 0.1446


In [99]:
results_pd.to_csv(os.path.join(PRED_DIR, 'ROC_PR_results.csv'), index=False)
print(f"Results saved -> {os.path.join(PRED_DIR, 'ROC_PR_results.csv')}")

Results saved -> E:\zcao\CA_Wildfire\Clean_Data\Model_Data\Evaluation\Features_w_Label_w_pred\Monthly_6_year_all_power\Calibration_Fixed_2001_2020\ROC_PR_results.csv
